# PS1 · Task 2 — Panel Return Prediction

Forecast daily returns for 100 firms; the test set is the **future** (`ret` withheld). The signal is
**momentum** + a **market factor** (`macro2`). Validation is time-aware, and time-series models are
scored by a **recursive** backtest (predictions fed forward, since lagged returns aren't in the test set).

Explore → models → ruled-out ideas → final recursive forecast → `task2_predictions.csv`.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
np.seterr(all="ignore")

train = pd.read_parquet("task2_training_data.parquet").sort_values(["firm_id", "date"]).reset_index(drop=True)
test  = pd.read_parquet("task2_testing_data.parquet")
EXO, CAT = ["macro2", "firm1", "firm2", "firm3", "price"], ["macro1"]
g = train.groupby("firm_id")
for k in range(1, 16): train[f"ret_lag{k}"]    = g["ret"].shift(k)
for k in range(1, 4):  train[f"macro2_lag{k}"] = g["macro2"].shift(k)

alphas = np.logspace(-3, 3, 40)
dates  = np.sort(train["date"].unique()); CUT = dates[-30]      # time-aware holdout = last 30 days

def ridge(num):
    return Pipeline([("pre", ColumnTransformer([("n", StandardScaler(), num),
                     ("c", OneHotEncoder(drop="first", handle_unknown="ignore"), CAT)])),
                     ("m", RidgeCV(alphas=alphas))])

def oracle(num):                       # validate with true lags (upper bound)
    d = train.dropna(subset=num); a, b = d[d.date < CUT], d[d.date >= CUT]
    return r2_score(b["ret"], ridge(num).fit(a[num+CAT], a["ret"]).predict(b[num+CAT]))

def recursive(num, k, lam=1.0):        # feed (lam * prediction) forward as the next lag (realistic)
    a = train[train.date < CUT].dropna(subset=num); m = ridge(num).fit(a[num+CAT], a["ret"])
    d = train.copy(); t = d["ret"].copy(); h = d.date >= CUT; d.loc[h, "ret"] = np.nan
    p = pd.Series(index=d.index, dtype=float)
    for dt in dates[dates >= CUT]:
        for j in range(1, k+1): d[f"ret_lag{j}"] = d.groupby("firm_id")["ret"].shift(j)
        r = d.date == dt; p.loc[r] = m.predict(d.loc[r, num+CAT]); d.loc[r, "ret"] = lam * p.loc[r].values
    return r2_score(t[h], p[h])

print("train", train.shape, "| test", test.shape, "| holdout = last 30 days")

## 1. Explore

In [ ]:
print("panel:", train.firm_id.nunique(), "firms x", train.date.nunique(), "days  |  test =",
      test.date.min().date(), "->", test.date.max().date(), "(future, ret withheld)")
print("contemporaneous corr with ret:", {c: round(train[c].corr(train.ret), 2) for c in EXO})
print("return autocorr (lags 1-3)   :", [round(train[f"ret_lag{k}"].corr(train.ret), 2) for k in (1, 2, 3)])
print("corr(price.pct_change, ret)  :", round(train.groupby("firm_id").price.pct_change().corr(train.ret), 2),
      "-> price is a noisy decoy")
print("mean ret by macro1 regime    :", train.groupby("macro1").ret.mean().round(3).to_dict())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3))
train.ret.hist(bins=50, ax=ax[0]); ax[0].set_title("daily ret")
train.groupby("date").ret.mean().plot(ax=ax[1], title="mean ret over time (crash -> recovery)")
pd.Series([train[f"ret_lag{k}"].corr(train.ret) for k in range(1, 8)], index=range(1, 8)).plot.bar(
    ax=ax[2], title="return autocorrelation"); plt.tight_layout(); plt.show()

## 2. Models

Time-aware holdout; `oracle` uses true lags (upper bound), `recursive` feeds predictions forward (what the future test set will see).

In [ ]:
def F(rl, ml=0):
    return [f"ret_lag{j}" for j in range(1, rl+1)] + [f"macro2_lag{j}" for j in range(1, ml+1)] + EXO

specs = [("cross-sectional (exog only)", EXO,     0),
         ("AR-X  ret-lags 1-3",          F(3),    3),
         ("window 5d",                   F(5),    5),
         ("window 10d",                  F(10),  10),
         ("window 15d",                  F(15),  15),
         ("window 5d + macro2 lags",     F(5, 3), 5)]
results = pd.DataFrame([{"model": n, "oracle_R2": round(oracle(num), 3),
                         "recursive_R2": round(recursive(num, k) if k else oracle(num), 3)}
                        for n, num, k in specs])
display(results)

**Momentum dominates** (cross-sectional 0.23 → AR-X ~0.55), a **5-day window + lagged `macro2`** is best (**recursive R² ~0.60**), and recursion costs little because `macro2` re-anchors each day.

## 3. Ruled out

Things that did **not** beat the best model (oracle R²) — kept brief.

In [ ]:
best = F(5, 3)
g2 = train.groupby("firm_id")
for f in ["firm1", "firm2", "firm3"]: train[f"{f}_lag1"] = g2[f].shift(1)
train["ret_roll5"] = g2["ret"].transform(lambda s: s.shift(1).rolling(5).mean())
train["m2_roll3"]  = g2["macro2"].transform(lambda s: s.rolling(3).mean())

def oracle_est(num, est):
    d = train.dropna(subset=num); a, b = d[d.date < CUT], d[d.date >= CUT]
    pre = ColumnTransformer([("n", StandardScaler(), num),
                             ("c", OneHotEncoder(drop="first", handle_unknown="ignore"), CAT)])
    return r2_score(b["ret"], Pipeline([("pre", pre), ("m", est)]).fit(a[num+CAT], a["ret"]).predict(b[num+CAT]))

print(f"best oracle = {oracle(best):.3f}  -- none of these beat it:")
print(f"  + firm-char lags    {oracle(best + ['firm1_lag1','firm2_lag1','firm3_lag1']):.3f}")
print(f"  + rolling-ret mean  {oracle(best + ['ret_roll5']):.3f}")
print(f"  + macro2 rolling    {oracle(best + ['m2_roll3']):.3f}")
print(f"  nonlinear HistGBM   {oracle_est(best, HistGradientBoostingRegressor(max_iter=400, max_depth=3, learning_rate=0.03, l2_regularization=1, random_state=0)):.3f}")
print(f"  recursion damping   {recursive(best, 5, 0.9):.3f}  (vs {recursive(best, 5):.3f} undamped -> damping hurts)")

d = train.dropna(subset=best); a, b = d[d.date < CUT], d[d.date >= CUT]; pr = pd.Series(index=b.index, dtype=float)
for rg, s in a.groupby("macro1"):
    sel = b.macro1 == rg
    if len(s) > 50 and sel.any():
        pr.loc[b.index[sel]] = Pipeline([("s", StandardScaler()), ("m", RidgeCV(alphas=alphas))]
                                         ).fit(s[best], s.ret).predict(b[sel][best])
print(f"  per-regime models   {r2_score(b.ret[pr.notna()], pr[pr.notna()]):.3f}  (overfits -> negative)")

## 4. Final model → `task2_predictions.csv`

In [ ]:
K = 5
featK = [f"ret_lag{j}" for j in range(1, K+1)] + ["macro2_lag1", "macro2_lag2", "macro2_lag3"] + EXO
fit_df = train.dropna(subset=featK)
final  = ridge(featK).fit(fit_df[featK+CAT], fit_df["ret"])

combo = pd.concat([train[["date","firm_id","ret"]+EXO+CAT].assign(_s="tr"),
                   test.assign(ret=np.nan)[["date","firm_id","ret"]+EXO+CAT].assign(_s="te")],
                  ignore_index=True).sort_values(["firm_id","date"]).reset_index(drop=True)
for k in range(1, 4): combo[f"macro2_lag{k}"] = combo.groupby("firm_id")["macro2"].shift(k)

pred = pd.Series(index=combo.index, dtype=float)
for dt in np.sort(test.date.unique()):
    for j in range(1, K+1): combo[f"ret_lag{j}"] = combo.groupby("firm_id")["ret"].shift(j)
    r = combo.date == dt; pred.loc[r] = final.predict(combo.loc[r, featK+CAT]); combo.loc[r, "ret"] = pred.loc[r].values

out = (combo[combo._s == "te"][["firm_id","date"]].assign(y_hat=pred[combo._s == "te"].values)
       .sort_values(["firm_id","date"]).reset_index(drop=True))
out.to_csv("task2_predictions.csv", index=False)
assert list(out.columns) == ["firm_id","date","y_hat"] and len(out) == len(test)
print("wrote task2_predictions.csv", out.shape); out.head()

## Conclusion

- **Best:** 5-day return window + lagged `macro2`, rolled forward recursively — realistic R² ≈ **0.60**.
- **Why:** strong momentum + a linear market factor; recursion is stable because `macro2` re-anchors it.
- **Dead ends:** firm-char lags, rolling means, interactions, fixed effects, damping, per-regime, and nonlinear models — none help. ~0.64 oracle is the noise ceiling.